In [1]:
import sqlite3
import pandas as pd
import os

In [2]:
DB_PATH = "../database/ecommerce.db"

os.makedirs("../database", exist_ok=True)

conn = sqlite3.connect(DB_PATH)

cursor = conn.cursor()

print("Connected Successfully!")

Connected Successfully!


In [3]:
customers = pd.read_csv("../data/cleaned/customers_clean.csv")

products = pd.read_csv("../data/cleaned/products_clean.csv")

orders = pd.read_csv("../data/cleaned/orders_clean.csv")

order_items = pd.read_csv("../data/cleaned/order_items_clean.csv")

print("Datasets Loaded")

Datasets Loaded


In [4]:
customers.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

products.to_sql(
    "products",
    conn,
    if_exists="replace",
    index=False
)

orders.to_sql(
    "orders",
    conn,
    if_exists="replace",
    index=False
)

order_items.to_sql(
    "order_items",
    conn,
    if_exists="replace",
    index=False
)

print("Tables Created Successfully")

Tables Created Successfully


In [5]:
tables = pd.read_sql_query(

"""
SELECT name
FROM sqlite_master
WHERE type='table';

""",

conn
)

tables

,name
0,customers
1,products
2,orders
3,order_items


In [6]:
for table in [

    "customers",
    "products",
    "orders",
    "order_items"

]:

    query = f"SELECT COUNT(*) AS Total FROM {table}"

    result = pd.read_sql_query(query, conn)

    print(table)

    print(result)

    print("-"*40)

customers
   Total
0    600
----------------------------------------
products
   Total
0    600
----------------------------------------
orders
   Total
0   1000
----------------------------------------
order_items
   Total
0   3000
----------------------------------------


**1. Total Revenue Per Category**

In [8]:
query = """

SELECT

    p.category,

    ROUND(

        SUM(

            oi.quantity *
            oi.unit_price *
            (1 - oi.discount_percent/100.0)

        ),

        2

    ) AS total_revenue

FROM order_items oi

JOIN products p

ON oi.product_id=p.product_id

GROUP BY p.category

ORDER BY total_revenue DESC;

"""

pd.read_sql_query(query, conn)

,category,total_revenue
0,Clothing,56432262.50
1,Electronics,55643356.09
2,Books,50574163.61
3,Home,42420085.12


**2. Top 10 Customers by Total Order Value**

In [9]:
query = """

SELECT

    c.customer_id,

    c.customer_name,

    ROUND(

        SUM(

            oi.quantity *
            oi.unit_price *
            (1 - oi.discount_percent / 100.0)

        ),

        2

    ) AS total_order_value

FROM customers c

JOIN orders o
ON c.customer_id = o.customer_id

JOIN order_items oi
ON o.order_id = oi.order_id

GROUP BY

    c.customer_id,
    c.customer_name

ORDER BY total_order_value DESC

LIMIT 10;

"""

top_customers = pd.read_sql_query(query, conn)

top_customers

,customer_id,customer_name,total_order_value
0,586,Heather Parrish,1915010.60
1,56,William Wilson,1788781.93
2,399,Mary Thompson,1710072.99
3,291,Jennifer Velasquez,1539620.46
4,510,David Hall,1523883.93
5,312,Ryan Rocha,1455637.38
6,591,Christopher Nielsen,1438371.53
7,527,Sarah Rivera,1402109.17
8,504,Michael Harris,1336611.41
9,215,Rachel Romero,1318964.47


**3. Month-wise Order Count (Last 12 Months)**

In [10]:
query = """

SELECT

    strftime('%Y-%m', order_date) AS order_month,

    COUNT(order_id) AS total_orders

FROM orders

WHERE order_date >= date('now', '-12 months')

GROUP BY order_month

ORDER BY order_month;

"""

monthly_orders = pd.read_sql_query(query, conn)

monthly_orders

,order_month,total_orders
0,2025-08,41
1,2025-09,40
2,2025-10,49
3,2025-11,29
4,2025-12,42
5,2026-01,46
6,2026-02,30
7,2026-03,49
8,2026-04,39
9,2026-05,55


**4. Find customers who placed orders but never had any item delivered**

In [11]:
query = """

SELECT DISTINCT

    c.customer_id,
    c.customer_name

FROM customers c

JOIN orders o
ON c.customer_id = o.customer_id

WHERE c.customer_id NOT IN (

    SELECT customer_id

    FROM orders

    WHERE status='DELIVERED'

);

"""

never_delivered = pd.read_sql_query(query, conn)

never_delivered

,customer_id,customer_name
0,1,Allison Hill
1,3,Cristian Santos
2,4,Abigail Shaffer
3,5,Gabrielle Davis
4,6,Monica Herrera
...,...,...
300,594,Kayla Jackson
301,595,James Rodriguez
302,596,Chase Davis
303,597,Kim Goodwin


**5. Products that were ordered but had more returns than purchases**

In [12]:
query = """

SELECT
    p.product_id,
    p.product_name,
    SUM(
        CASE
            WHEN oi.quantity>0
            THEN oi.quantity
            ELSE 0
        END
    ) AS purchased,
    ABS(
        SUM(
            CASE
                WHEN oi.quantity<0
                THEN oi.quantity
                ELSE 0
            END
        )
    ) AS returned
FROM products p
JOIN order_items oi
ON p.product_id=oi.product_id
GROUP BY
    p.product_id,
    p.product_name
HAVING returned > purchased;

"""

returned_products = pd.read_sql_query(query, conn)

returned_products

,product_id,product_name,purchased,returned
0,31,Jeans,2,5
1,189,Dress,1,2
2,471,The Silent Patient,3,4
3,478,Jeans,0,5
4,494,Realme Gt,0,4
5,522,Samsung Galaxy S24,2,3


**6. Return Rate Per Category**

In [13]:
query = """

SELECT

    p.category,

    ROUND(

        SUM(

            CASE

                WHEN oi.quantity<0

                THEN ABS(oi.quantity)

                ELSE 0

            END

        ) *100.0

        /

        SUM(ABS(oi.quantity)),

        2

    ) AS return_rate_percent

FROM products p

JOIN order_items oi

ON p.product_id=oi.product_id

GROUP BY p.category

ORDER BY return_rate_percent DESC;

"""

return_rate = pd.read_sql_query(query, conn)

return_rate

,category,return_rate_percent
0,Electronics,3.70
1,Books,3.10
2,Clothing,2.61
3,Home,2.51


**7. Running Totals with Window Functions**

In [14]:
query="""
SELECT
o.region_code,
DATE(o.order_date) AS order_date,
ROUND(SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)),2) AS daily_revenue,
ROUND(SUM(SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0))) OVER(PARTITION BY o.region_code ORDER BY DATE(o.order_date)),2) AS running_total
FROM orders o
JOIN order_items oi ON o.order_id=oi.order_id
GROUP BY o.region_code,DATE(o.order_date)
ORDER BY o.region_code,DATE(o.order_date);
"""
pd.read_sql_query(query,conn)

,region_code,order_date,daily_revenue,running_total
0,EAST,2024-08-09,74738.97,74738.97
1,EAST,2024-08-12,115197.46,189936.43
2,EAST,2024-08-15,301816.83,491753.26
3,EAST,2024-08-16,37087.68,528840.94
4,EAST,2024-08-20,60799.29,589640.23
...,...,...,...,...
800,WEST,2026-07-10,646195.70,46300443.49
801,WEST,2026-07-18,15433.97,46315877.47
802,WEST,2026-07-24,171657.85,46487535.32
803,WEST,2026-07-31,408565.34,46896100.66


**8. For each category, rank products by total revenue.**

**Products having the same revenue should receive the same rank.**

In [17]:
query="""
SELECT
category,
product_name,
total_revenue,
DENSE_RANK() OVER(PARTITION BY category ORDER BY total_revenue DESC) AS rank_in_category
FROM(
SELECT
p.category,
p.product_name,
ROUND(SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)),2) AS total_revenue
FROM products p
JOIN order_items oi ON p.product_id=oi.product_id
GROUP BY p.category,p.product_name
)t
ORDER BY category,rank_in_category;
"""
pd.read_sql_query(query,conn)

,category,product_name,total_revenue,rank_in_category
0,Books,The Alchemist,5692878.28,1
1,Books,Python Programming,4641917.33,2
2,Books,Long Walk To Freedom,4133711.80,3
3,Books,Operating System,3432498.77,4
4,Books,Becoming,3389539.27,5
...,...,...,...,...
65,Home,Mixer Grinder,1451973.82,14
66,Home,Study Table,1388732.07,15
67,Home,Bed,1357069.56,16
68,Home,Bookshelf,1322894.59,17


**9. For each customer calculate:**

customer_id
order_date
previous_order_date
days_gap

Flag customers whose average gap is greater than 30 days as At Risk.

In [18]:
query="""
WITH customer_orders AS(
SELECT
customer_id,
DATE(order_date) AS order_date,
LAG(DATE(order_date)) OVER(PARTITION BY customer_id ORDER BY DATE(order_date)) AS previous_order_date
FROM orders
WHERE customer_id!=-1
),
gap_analysis AS(
SELECT
customer_id,
order_date,
previous_order_date,
JULIANDAY(order_date)-JULIANDAY(previous_order_date) AS days_gap
FROM customer_orders
)
SELECT
customer_id,
order_date,
previous_order_date,
ROUND(days_gap,0) AS days_gap,
CASE
WHEN AVG(days_gap) OVER(PARTITION BY customer_id)>30 THEN 'At Risk'
ELSE 'Active'
END AS customer_status
FROM gap_analysis
ORDER BY customer_id,order_date;
"""
pd.read_sql_query(query,conn)

,customer_id,order_date,previous_order_date,days_gap,customer_status
0,1,2025-02-28,NaN,NaN,At Risk
1,1,2026-02-26,2025-02-28,363.0,At Risk
2,3,2025-03-26,NaN,NaN,Active
3,4,2024-08-15,NaN,NaN,At Risk
4,4,2025-08-11,2024-08-15,361.0,At Risk
...,...,...,...,...,...
945,599,2026-03-29,NaN,NaN,Active
946,600,2025-05-10,NaN,NaN,At Risk
947,600,2025-06-06,2025-05-10,27.0,At Risk
948,600,2025-10-14,2025-06-06,130.0,At Risk


**10. Multi-Level CTE**

Requirement

- Calculate monthly revenue per customer.
- Categorize customers:
High (>10000), Medium (5000–10000), Low (<5000)
- Show the count of customers in each category per month.

In [19]:
query="""
WITH monthly_revenue AS(
SELECT
o.customer_id,
strftime('%Y-%m',o.order_date) AS order_month,
SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) AS revenue
FROM orders o
JOIN order_items oi ON o.order_id=oi.order_id
WHERE o.customer_id!=-1
GROUP BY o.customer_id,strftime('%Y-%m',o.order_date)
),
customer_category AS(
SELECT
customer_id,
order_month,
revenue,
CASE
WHEN revenue>10000 THEN 'High'
WHEN revenue BETWEEN 5000 AND 10000 THEN 'Medium'
ELSE 'Low'
END AS customer_segment
FROM monthly_revenue
)
SELECT
order_month,
customer_segment,
COUNT(customer_id) AS total_customers
FROM customer_category
GROUP BY order_month,customer_segment
ORDER BY order_month,customer_segment;
"""
pd.read_sql_query(query,conn)

,order_month,customer_segment,total_customers
0,2024-08,High,32
1,2024-08,Low,1
2,2024-09,High,37
3,2024-10,High,35
4,2024-10,Low,1
5,2024-11,High,40
6,2024-11,Low,2
7,2024-12,High,46
8,2024-12,Low,2
9,2025-01,High,36


**11 – NTILE() Customer Segmentation**

Requirement

Divide customers into 4 quartiles based on lifetime value.

Labels:

Platinum
Gold
Silver
Bronze

In [20]:
query="""
WITH customer_value AS(
SELECT
o.customer_id,
ROUND(SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)),2) AS total_value
FROM orders o
JOIN order_items oi ON o.order_id=oi.order_id
WHERE o.customer_id!=-1
GROUP BY o.customer_id
)
SELECT
customer_id,
total_value,
NTILE(4) OVER(ORDER BY total_value DESC) AS quartile,
CASE NTILE(4) OVER(ORDER BY total_value DESC)
WHEN 1 THEN 'Platinum'
WHEN 2 THEN 'Gold'
WHEN 3 THEN 'Silver'
ELSE 'Bronze'
END AS quartile_label
FROM customer_value;
"""
pd.read_sql_query(query,conn)

,customer_id,total_value,quartile,quartile_label
0,586,1915010.60,1,Platinum
1,56,1788781.93,1,Platinum
2,399,1710072.99,1,Platinum
3,291,1539620.46,1,Platinum
4,510,1523883.93,1,Platinum
...,...,...,...,...
462,255,-45180.60,4,Bronze
463,150,-62674.42,4,Bronze
464,466,-80592.56,4,Bronze
465,49,-105343.04,4,Bronze


**12 – Year-over-Year Revenue Comparison**

Requirement

Compare each month's revenue with the same month in the previous year.

In [21]:
query="""
WITH monthly_sales AS(
SELECT
strftime('%Y',o.order_date) AS year,
strftime('%m',o.order_date) AS month,
ROUND(SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)),2) AS revenue
FROM orders o
JOIN order_items oi ON o.order_id=oi.order_id
GROUP BY year,month
)
SELECT
year,
month,
revenue,
LAG(revenue,1) OVER(PARTITION BY month ORDER BY year) AS prev_year_revenue,
ROUND(((revenue-LAG(revenue,1) OVER(PARTITION BY month ORDER BY year))*100.0)/LAG(revenue,1) OVER(PARTITION BY month ORDER BY year),2) AS yoy_growth_percent
FROM monthly_sales
ORDER BY year,month;
"""
pd.read_sql_query(query,conn)

,year,month,revenue,prev_year_revenue,yoy_growth_percent
0,2024,08,7090245.26,NaN,NaN
1,2024,09,10144585.39,NaN,NaN
2,2024,10,9578463.33,NaN,NaN
3,2024,11,9137677.73,NaN,NaN
4,2024,12,10434970.18,NaN,NaN
5,2025,01,7918122.77,NaN,NaN
6,2025,02,6594131.78,NaN,NaN
7,2025,03,5915400.36,NaN,NaN
8,2025,04,6095081.32,NaN,NaN
9,2025,05,10846388.98,NaN,NaN


**13 – First/Last Purchased Category Analysis**

In [22]:
query="""
WITH customer_history AS(
SELECT
o.customer_id,
DATE(o.order_date) AS order_date,
p.category,
ROW_NUMBER() OVER(PARTITION BY o.customer_id ORDER BY DATE(o.order_date)) AS first_order,
ROW_NUMBER() OVER(PARTITION BY o.customer_id ORDER BY DATE(o.order_date) DESC) AS last_order
FROM orders o
JOIN order_items oi ON o.order_id=oi.order_id
JOIN products p ON oi.product_id=p.product_id
WHERE o.customer_id!=-1
),
first_category AS(
SELECT customer_id,category AS first_category
FROM customer_history
WHERE first_order=1
),
last_category AS(
SELECT customer_id,category AS last_category
FROM customer_history
WHERE last_order=1
)
SELECT
f.customer_id,
f.first_category,
l.last_category,
CASE
WHEN f.first_category=l.last_category THEN 'No'
ELSE 'Yes'
END AS category_shift
FROM first_category f
JOIN last_category l ON f.customer_id=l.customer_id
ORDER BY f.customer_id;
"""
pd.read_sql_query(query,conn)

,customer_id,first_category,last_category,category_shift
0,1,Electronics,Electronics,No
1,3,Books,Books,No
2,4,Home,Books,Yes
3,5,Books,Clothing,Yes
4,6,Books,Books,No
...,...,...,...,...
462,595,Electronics,Home,Yes
463,596,Home,Books,Yes
464,597,Books,Books,No
465,599,Books,Books,No


**14. Calculate what percentage of total revenue comes from top customers.**

In [23]:
query="""
WITH customer_revenue AS(
SELECT
o.customer_id,
ROUND(SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)),2) AS revenue
FROM orders o
JOIN order_items oi ON o.order_id=oi.order_id
WHERE o.customer_id!=-1
GROUP BY o.customer_id
)
SELECT
customer_id,
revenue,
SUM(revenue) OVER(ORDER BY revenue DESC) AS cumulative_revenue,
ROUND(SUM(revenue) OVER(ORDER BY revenue DESC)*100.0/SUM(revenue) OVER(),2) AS cumulative_percent
FROM customer_revenue
ORDER BY revenue DESC;
"""
pd.read_sql_query(query,conn)

,customer_id,revenue,cumulative_revenue,cumulative_percent
0,586,1915010.60,1.915011e+06,0.99
1,56,1788781.93,3.703793e+06,1.91
2,399,1710072.99,5.413866e+06,2.79
3,291,1539620.46,6.953486e+06,3.59
4,510,1523883.93,8.477370e+06,4.38
...,...,...,...,...
462,255,-45180.60,1.940844e+08,100.20
463,150,-62674.42,1.940217e+08,100.16
464,466,-80592.56,1.939411e+08,100.12
465,49,-105343.04,1.938358e+08,100.07


**15. Group customers by registration month and calculate Month-0 to Month-3 retention.**

In [24]:
query="""
WITH customer_cohort AS(
SELECT
customer_id,
strftime('%Y-%m',registration_date) AS cohort_month
FROM customers
),
customer_orders AS(
SELECT
o.customer_id,
strftime('%Y-%m',o.order_date) AS order_month,
c.cohort_month,
((CAST(substr(strftime('%Y-%m',o.order_date),1,4) AS INTEGER)-CAST(substr(c.cohort_month,1,4) AS INTEGER))*12+
(CAST(substr(strftime('%Y-%m',o.order_date),6,2) AS INTEGER)-CAST(substr(c.cohort_month,6,2) AS INTEGER))) AS month_number
FROM orders o
JOIN customer_cohort c ON o.customer_id=c.customer_id
)
SELECT
cohort_month,
COUNT(DISTINCT CASE WHEN month_number=0 THEN customer_id END) AS month0,
COUNT(DISTINCT CASE WHEN month_number=1 THEN customer_id END) AS month1,
COUNT(DISTINCT CASE WHEN month_number=2 THEN customer_id END) AS month2,
COUNT(DISTINCT CASE WHEN month_number=3 THEN customer_id END) AS month3
FROM customer_orders
GROUP BY cohort_month
ORDER BY cohort_month;
"""
pd.read_sql_query(query,conn)

,cohort_month,month0,month1,month2,month3
0,2023-08,0,0,0,0
1,2023-09,0,0,0,0
2,2023-10,0,0,0,0
3,2023-11,0,0,0,0
4,2023-12,0,0,0,0
5,2024-01,0,0,0,0
6,2024-02,0,0,0,0
7,2024-03,0,0,0,0
8,2024-04,0,0,0,0
9,2024-05,0,0,0,2


**16. Compare each customer's current order with their previous order.**

In [25]:
query="""
WITH customer_orders AS(
SELECT
customer_id,
order_id,
DATE(order_date) AS order_date,
LAG(order_id) OVER(PARTITION BY customer_id ORDER BY DATE(order_date)) AS previous_order_id,
LAG(DATE(order_date)) OVER(PARTITION BY customer_id ORDER BY DATE(order_date)) AS previous_order_date
FROM orders
WHERE customer_id!=-1
)
SELECT
customer_id,
order_id,
order_date,
previous_order_id,
previous_order_date,
ROUND(JULIANDAY(order_date)-JULIANDAY(previous_order_date),0) AS days_between_orders
FROM customer_orders
ORDER BY customer_id,order_date;
"""
pd.read_sql_query(query,conn)

,customer_id,order_id,order_date,previous_order_id,previous_order_date,days_between_orders
0,1,764,2025-02-28,NaN,NaN,NaN
1,1,458,2026-02-26,764.0,2025-02-28,363.0
2,3,915,2025-03-26,NaN,NaN,NaN
3,4,590,2024-08-15,NaN,NaN,NaN
4,4,766,2025-08-11,590.0,2024-08-15,361.0
...,...,...,...,...,...,...
945,599,741,2026-03-29,NaN,NaN,NaN
946,600,231,2025-05-10,NaN,NaN,NaN
947,600,578,2025-06-06,231.0,2025-05-10,27.0
948,600,913,2025-10-14,578.0,2025-06-06,130.0
